# tsgen — Full Training Runs on Colab

This notebook runs the diffusion/VAE experiments on a Colab GPU. Baselines (multivariate Gaussian, stationary block bootstrap, CCC-GARCH) are fast enough to run locally; they are excluded here by default.

**Prerequisite (one-time):** copy `~/.finbase/timeseries.db` from your local machine to Google Drive at `My Drive/tsgen/timeseries.db`. See the README in `notebooks/README.md` for details.

Artifacts (model weights, logs, plots) are written to `My Drive/tsgen/runs/<exp_name>/`. You can close and resume the notebook between experiments — Drive state survives runtime disconnects.

## 1. Runtime check

In [ ]:
# Confirm GPU is allocated. Runtime → Change runtime type → T4 GPU.
import torch
assert torch.cuda.is_available(), 'No GPU allocated. Runtime → Change runtime type → GPU (T4).'
print(f'CUDA device: {torch.cuda.get_device_name(0)}')
print(f'torch version: {torch.__version__}')

## 2. Mount Google Drive

Expected Drive layout:

```
My Drive/tsgen/
  ├── timeseries.db         (copy from ~/.finbase/timeseries.db)
  └── runs/                 (artifacts land here; created automatically)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/tsgen'
DB_ON_DRIVE = f'{DRIVE_BASE}/timeseries.db'
RUNS_DIR = f'{DRIVE_BASE}/runs'

assert os.path.exists(DB_ON_DRIVE), (
    f'Database not found at {DB_ON_DRIVE}. Upload timeseries.db to '
    f'My Drive/tsgen/ before running.'
)
os.makedirs(RUNS_DIR, exist_ok=True)
print(f'Drive OK. DB size: {os.path.getsize(DB_ON_DRIVE) / (1024**2):.1f} MB')

## 3. Copy the DB to Colab local disk

SQLite over Drive-mount is ~10× slower than local disk. Copy once (a few seconds) so every query thereafter is fast.

In [ ]:
import shutil, os
LOCAL_DB = '/root/.finbase/timeseries.db'
os.makedirs(os.path.dirname(LOCAL_DB), exist_ok=True)
if not os.path.exists(LOCAL_DB) or os.path.getmtime(LOCAL_DB) < os.path.getmtime(DB_ON_DRIVE):
    print('Copying DB to Colab local disk...')
    shutil.copy(DB_ON_DRIVE, LOCAL_DB)
    print(f'Done. Local copy: {os.path.getsize(LOCAL_DB) / (1024**2):.1f} MB')
else:
    print('Local DB already up to date.')

# Configure finbase to read from the local copy
with open('/root/.finbaserc', 'w') as f:
    f.write(f'database:\n  path: {LOCAL_DB}\n')
print('Wrote /root/.finbaserc')

## 4. Install dependencies

`finbase` is from PyPI; `tsgen` is cloned from GitHub (develop branch).

In [ ]:
!pip install --quiet finbase arch pydantic pyyaml statsmodels scikit-learn tqdm mlflow 2>&1 | tail -5

In [ ]:
import os, subprocess
REPO_DIR = '/content/tsgen'
REPO_URL = 'https://github.com/shoom1/tsgen.git'
BRANCH = 'develop'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH, '--quiet'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', '--quiet'], check=True)

# Editable install
!pip install --quiet -e {REPO_DIR} 2>&1 | tail -3

In [ ]:
# Verify package imports and GPU visibility
from tsgen.models.registry import ModelRegistry
from finbase import DataClient
import torch

print('Registered models:', sorted(ModelRegistry.list_models().keys()))
print()
client = DataClient()
sample = client.get_data(symbols=['AAPL'], start='2024-01-01', end='2024-12-31',
                         columns=['adj_close'], format='wide')
print(f'DataClient smoke test: loaded {len(sample)} rows for AAPL 2024.')
print(f'CUDA visible: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU"})')

## 5. Generate standardized experiment configs

Experiment configs live in `experiments/` which is gitignored (they're personal work/analysis, not shared code). We regenerate them from the canonical source-of-truth script.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', f'{REPO_DIR}/scripts/generate_standard_configs.py'],
    cwd=REPO_DIR, check=True, capture_output=True, text=True,
)
print(result.stdout)

## 6. Run experiments

The cell below launches each selected experiment via the `tsgen` CLI, overriding `output_dir` so artifacts land in `<DRIVE>/runs/<exp_name>/`. Disable an experiment by commenting it out of `EXPERIMENTS`.

Each diffusion experiment is 200 epochs on 100 tickers. Rough T4 estimates (DDPM 500 timesteps, batch 32):

| exp | model | expected T4 time |
|---|---|---|
| 0001 | TimeVAE (100 epochs) | ~30-45 min |
| 0002 | UNet1D | ~45-60 min |
| 0003 | UNet1D (data_fix) | ~45-60 min |
| 0004 | DiffusionTransformer | ~45-60 min |
| 0006 | MambaDiffusion | ~90-120 min (largest) |
| 0008 | DiffWave1D | ~30-45 min |
| 0009 | DiT1D | ~45-60 min |

Free-tier T4 gives ~12h/day. If you need to span multiple sessions, run a subset, then come back and re-run this cell — experiments that already have `model_final.pt` in their Drive runs folder can be skipped manually by commenting them out.

In [ ]:
# Which experiments to run. Comment out any you've already completed.
EXPERIMENTS = [
    ('0001_timevae', 'experiments/0001_timevae_all_stocks/config.yaml'),
    ('0002_unet', 'experiments/0002_unet_all_stocks/config.yaml'),
    ('0003_unet_fix', 'experiments/0003_unet_data_fix/config.yaml'),
    ('0004_transformer', 'experiments/0004_transformer_all_stocks/config.yaml'),
    ('0006_mamba', 'experiments/0006_mamba_default/config.yaml'),
    ('0008_diffwave', 'experiments/0008_diffwave/config.yaml'),
    ('0009_dit', 'experiments/0009_dit/config.yaml'),
]

In [ ]:
import subprocess, time, os

for name, config_relpath in EXPERIMENTS:
    out_dir = f'{RUNS_DIR}/{name}'
    os.makedirs(out_dir, exist_ok=True)
    cmd = [
        'tsgen',
        '--config', config_relpath,
        '--mode', 'train_eval',
        '--override', f'output_dir={out_dir}',
        # Train on GPU
        '--override', 'device=cuda',
    ]
    print(f'=== {name} ===')
    print(' '.join(cmd))
    t0 = time.perf_counter()
    r = subprocess.run(cmd, cwd=REPO_DIR)
    elapsed = time.perf_counter() - t0
    status = 'PASS' if r.returncode == 0 else f'FAIL ({r.returncode})'
    print(f'--- {name}: {status} in {elapsed/60:.1f} min ---')
    print()

## 7. Inspect artifacts on Drive

Artifacts land under `My Drive/tsgen/runs/<exp_name>/`. You can browse them in the Drive web UI or iterate here.

In [ ]:
import os
for name in sorted(os.listdir(RUNS_DIR)):
    path = f'{RUNS_DIR}/{name}'
    if not os.path.isdir(path):
        continue
    entries = sorted(os.listdir(path))
    print(f'{name}: {entries}')

## 8. (Optional) Smoke test

To verify everything is wired correctly at a tiny budget before committing to the full runs (useful after a code change in `develop`):

In [ ]:
import subprocess
r = subprocess.run(['python', f'{REPO_DIR}/scripts/smoke_test_experiments.py'],
                   cwd=REPO_DIR)
print(f'smoke exit code: {r.returncode}')